# TD05 — Support Vector Machines

> [!abstract] Objectif
> Le cours a posé la question : *parmi tous les séparateurs qui marchent, lequel choisir ?*
> Le SVM répond : **le séparateur à marge maximale**. Dans ce TD on va :
>
> 1. manipuler la **marge** et les **vecteurs supports** à la main ;
> 2. constater que deux runs du perceptron donnent des séparateurs différents — et apprendre
>    à prédire par la marge lequel généralise le mieux ;
> 3. **implémenter** un SVM approché (marge souple : hinge + régularisation) ;
> 4. faire jouer le compromis quand les données sont bruitées ;
> 5. traiter des données non séparables en **projetant** avec $\Phi$ (cf. partie 4) — et voir
>    où les noyaux (partie 9) feront mieux.

Cours correspondant : module « 5 - Support Vector Machines ».

> [!note] Changement de convention (important)
> Jusqu'ici les cibles étaient $t \in \{0, 1\}$ et le biais vivait dans le poids $w_0$
> (TD01–04). Le SVM demande une écriture symétrique :
>
> | avant (TD01–04) | dès maintenant |
> |---|---|
> | $t \in \{0,1\}$ | $t \in \{-1,+1\}$ (conversion : $t = 2\,t_{01} - 1$) |
> | pré-activation $= w^\top x - w_0$ | pré-activation $z(x) = w^\top x + b$ (biais `b`) |
> | prédire $\{0,1\}$ | prédire $\text{sign}(z) \in \{\pm 1\}$ |
>
> C'est *la même famille de modèles* : c'est le codage qui change, pour que la marge
> s'écrive joliment ($t_i(w^\top x_i + b) \geq 1$).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass


def make_axis(xlim=(-2.6, 2.6), ylim=(-2.6, 2.6)):
    fig, ax = plt.subplots(figsize=(5.4, 4.4))
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_aspect('equal', adjustable='box')
    ax.axhline(0, color='black', lw=0.8)
    ax.axvline(0, color='black', lw=0.8)
    ax.grid(alpha=0.3)
    return fig, ax


def plot_data(ax, X, t, label=True):
    """Nuage de points : carré bleu pour la classe -1, disque orange pour la +1."""
    m1, m2 = np.asarray(t) == -1, np.asarray(t) == 1
    ax.scatter(*X[m1].T, s=45, c='tab:blue', marker='s', zorder=3,
               label='classe $-1$' if label else None)
    ax.scatter(*X[m2].T, s=45, c='tab:orange', marker='o', zorder=3,
               label='classe $+1$' if label else None)


def display_separator(ax, w, b, **kw):
    """Droite $w^\\top x + b = 0$ — robuste aux droites verticales."""
    w = np.asarray(w, dtype=float)
    n = np.linalg.norm(w)
    c = -b * w / n
    d = np.array([-w[1], w[0]]) / n
    p1, p2 = c + 9 * d, c - 9 * d
    style = dict(color='black', lw=2.0, zorder=2)
    style.update(kw)
    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], **style)


def shade_marge(ax, w, b, m):
    """Bande comprise entre $w^\\top x + b = -m$ et $w^\\top x + b = +m$ (w normé)."""
    w = np.asarray(w, dtype=float)
    w = w / np.linalg.norm(w)
    d = np.array([-w[1], w[0]])
    base = -b * w
    poly = np.array([
        base + m * w + 9 * d,
        base + m * w - 9 * d,
        base - m * w - 9 * d,
        base - m * w + 9 * d,
    ])
    ax.fill(poly[:, 0], poly[:, 1], color='gray', alpha=0.13, zorder=1)


def marge_separateur(w, b, X, t):
    """Outil fourni : marge géométrique = min_i [ t_i (w^T x_i + b) ] / ||w||."""
    return float((np.asarray(t) * (np.asarray(X) @ w + b)).min() / np.linalg.norm(w))


def accuracy(t, y):
    """Outil fourni : proportion de prédictions correctes."""
    return float(np.mean(np.asarray(t) == np.asarray(y)))


def cercle(ax, x, r=0.22, **kw):
    """Cercle de mise en évidence autour d'un point."""
    style = dict(fill=False, lw=2.4, zorder=5, color='black')
    style.update(kw)
    ax.add_patch(plt.Circle(np.asarray(x, dtype=float), r, **style))


print('bibliothèques prêtes — numpy', np.__version__)

## 1. La marge à la main

On se donne 6 points (deux amas serrés) et trois séparateurs **tous valides** (ils classent
tout le monde correctement) :

- $s_1$ : $w = (1,\ 0)$, $b = 0$ (droite verticale $x_1 = 0$) ;
- $s_2$ : $w = (1,\ 0.25)$, $b = 0$ (inclinée douce) ;
- $s_3$ : $w = (1,\ 0.8)$, $b = 0$ (très inclinée).

> **Exercice 1** — Pour chaque séparateur, calculez **à la main** :
> 1. la marge : $\;m = \min_i \dfrac{t_i\,(w^\top x_i + b)}{\|w\|}$ ;
> 2. le classement des trois séparateurs, de la plus large marge à la plus petite ;
> 3. quels points sont les **vecteurs supports** (ceux qui touchent la bordure de bande).
>
> Complétez le tableau, puis vérifiez avec la cellule corrigée.

In [ ]:
X_ex1 = np.array([
    [-2.0, 0.0], [-1.5, 0.5], [-1.5, -0.5],   # classe -1
    [ 2.0, 0.0], [ 1.5, 0.5], [ 1.5, -0.5],   # classe +1
])
t_ex1 = np.array([-1, -1, -1, 1, 1, 1])

separateurs = {
    's1 (verticale)':     (np.array([1.0, 0.0]),  0.0),
    's2 (inclinée)':      (np.array([1.0, 0.25]), 0.0),
    's3 (très inclinée)': (np.array([1.0, 0.8]),  0.0),
}

fig, ax = make_axis((-3.0, 3.0), (-1.7, 1.7))
plot_data(ax, X_ex1, t_ex1)
for name, (w, b) in separateurs.items():
    display_separator(ax, w, b, lw=1.2)
ax.legend(fontsize=8, loc='lower right')
ax.set_title('les trois séparateurs sur le même nuage')
plt.show()

In [ ]:
# BEGIN
# Corrigé : marge calculée avec l'outil fourni, bande colorée, vecteurs supports cerclés.
for name, (w, b) in separateurs.items():
    m = marge_separateur(w, b, X_ex1, t_ex1)
    fig, ax = make_axis((-3.0, 3.0), (-1.7, 1.7))
    shade_marge(ax, w, b, m)
    display_separator(ax, w, b)
    plot_data(ax, X_ex1, t_ex1)
    zx = t_ex1 * (X_ex1 @ w + b) / np.linalg.norm(w)   # distance algébrique signée de chaque point
    sup = np.where(np.isclose(zx, m))[0]               # les points posés sur la bordure de bande
    for i in sup:
        cercle(ax, X_ex1[i])
    ax.set_title(f'{name} — marge $\\approx$ {m:.3f}')
    plt.show()
    print(f'{name} : marge = {m:.4f}   vecteurs supports = '
          f'{[(float(xi), float(yi)) for xi, yi in X_ex1[sup]]}')
# END

In [ ]:
m1 = marge_separateur(separateurs['s1 (verticale)'][0],     separateurs['s1 (verticale)'][1],     X_ex1, t_ex1)
m2 = marge_separateur(separateurs['s2 (inclinée)'][0],      separateurs['s2 (inclinée)'][1],      X_ex1, t_ex1)
m3 = marge_separateur(separateurs['s3 (très inclinée)'][0], separateurs['s3 (très inclinée)'][1], X_ex1, t_ex1)

# m1 : les points les plus proches sont en x1 = ±1.5, distance 1.5 / ||w|| = 1.5 / 1
# m2 : le plus proche est (-1.5, 0.5) : | -1.5 + 0.25 * 0.5 | / ||(1, 0.25)|| = 1.375 / 1.0308
# m3 : le plus proche est (-1.5, 0.5) : | -1.5 + 0.8  * 0.5 | / ||(1, 0.8)||  = 1.1   / 1.2806
assert np.isclose(m1, 1.5, atol=1e-6), f'attendu 1.5, obtenu {m1}'
assert np.isclose(m2, 1.3337, atol=1e-3), f'attendu 1.3337, obtenu {m2}'
assert np.isclose(m3, 0.8590, atol=1e-3), f'attendu 0.8590, obtenu {m3}'
assert m1 > m2 > m3, 's1 doit avoir la plus large marge'
print('marge check — OK')

> [!note] Ce qu'il faut voir
> Les trois séparateurs sont *parfaitement corrects* sur ces données. Pourtant leurs marges
> sont très différentes ($1.5 > 1.33 > 0.86$), et les **vecteurs supports changent de
> séparateur en séparateur** (colonne « supports » impressionnée de votre tableau). La marge
> est une mesure du *séparateur*, pas des données : c'est elle qu'on va maximiser.

## 2. Deux runs, deux séparateurs

> **Exercice 2** — Données partagées par tous les runs : 30 points d'entraînement + 30 points
> de test, séparés par la même droite (pente 0.5, sans biais). Le jeu de *test* autorise des
> points beaucoup plus proches de la frontière que le jeu d'*entraînement*
> (cf. figure de la cellule suivante).
>
> 1. Entraînez deux `Perceptron` à partir de $w = (0, 0)$, $b = 0$ sur **le même** jeu
>    d'entraînement ; seul change l'ordre de présentation des points :
>    `rng = 3` pour le run 1, `rng = 6` pour le run 2.
> 2. Affichez les deux frontières ; calculez la marge de chacune **sur le jeu
>    d'entraînement** (outil `marge_separateur`).
> 3. Comparez erreurs d'entraînement et erreurs de test pour les deux runs.
>
> Ce que vous devez observer : deux séparateurs parfaits en entraînement, mais des marges
> différentes et des erreurs de test différentes — la marge prédit le gagnant.

In [ ]:
def make_dataset(w1, w2, b, n, marge_min, seed):
    """Outil fourni : n points uniformes dans [-2.2, 2.2]^2, séparés par la droite (w1, w2, b).

    On rejette tout point tombant dans la bande |w^T x + b| < marge_min : le jeu
    reste « propre » autour de la frontière. Cibles t = -1 d'un côté, +1 de l'autre.
    """
    g = np.random.default_rng(seed)
    X, t = [], []
    while len(X) < n:
        x = g.uniform(-2.2, 2.2, size=2)
        z = w1 * x[0] + w2 * x[1] + b
        if abs(z) >= marge_min:
            X.append(x)
            t.append(1 if z > 0 else -1)
    return np.array(X), np.array(t)


X_train, t_train = make_dataset(1.0, 0.5, 0.0, n=30, marge_min=0.55, seed=42)
X_test, t_test = make_dataset(1.0, 0.5, 0.0, n=30, marge_min=0.15, seed=43)

fig, axes = plt.subplots(1, 2, figsize=(10.6, 4.2), sharex=True, sharey=True)
plot_data(axes[0], X_train, t_train)
display_separator(axes[0], np.array([1.0, 0.5]), 0.0)
axes[0].set_title('entraînement (aucun point à moins de 0.55 de la droite)')
plot_data(axes[1], X_test, t_test)
display_separator(axes[1], np.array([1.0, 0.5]), 0.0)
axes[1].set_title('test (des points très proches de la droite)')
for ax in axes:
    ax.set_xlim(-2.6, 2.6)
    ax.set_ylim(-2.6, 2.6)
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.set_aspect('equal')
    ax.grid(alpha=0.3)
plt.show()

> [!info] Le perceptron, recodé en $\pm 1$
> Même algorithme qu'au TD01, deux retouches d'écriture :
> - le biais est un champ séparé `b` (fini le $w_0$) : $z(x) = w^\top x + b$ ;
> - les cibles sont $\pm 1$ : « mal classé » se dit $t_i\, z(x_i) \leq 0$, et la mise à jour
>   devient $w \mathrel{+}= r\,t_i\,x_i$ puis $b \mathrel{+}= r\,t_i$.
>
> Les époques mélangent les points — c'est le `rng` passé à `fit` qui décide de l'ordre.

In [ ]:
@dataclass
class Perceptron:
    """Perceptron à cibles ±1 (recodé du TD01) — code redonné, ne pas réécrire.

    z(x) = w^T x + b ; prédiction = signe de z. Mise à jour quand t_i z(x_i) <= 0 :
    w += lr * t_i * x_i, puis b += lr * t_i.
    """
    w: np.ndarray
    b: float = 0.0
    lr: float = 0.1

    def z(self, x):
        return float(self.w @ x + self.b)

    def output(self, x):
        return 1 if self.z(x) > 0 else -1

    def fit(self, X, t, epochs=100, rng=0):
        """Entraînement échantillon par échantillon ; l'ordre est mélangé par `rng`."""
        g = np.random.default_rng(rng)
        n = len(X)
        for _ in range(epochs):
            moved = False
            for i in g.permutation(n):
                if t[i] * self.z(X[i]) <= 0:
                    self.w = self.w + self.lr * t[i] * X[i]
                    self.b = self.b + self.lr * t[i]
                    moved = True
            if not moved:
                break
        return self

In [ ]:
# BEGIN
p1 = Perceptron(np.zeros(2)).fit(X_train, t_train, rng=3)
p2 = Perceptron(np.zeros(2)).fit(X_train, t_train, rng=6)

fig, axes = plt.subplots(1, 2, figsize=(10.6, 4.2), sharex=True, sharey=True)
for ax, p, nom in ((axes[0], p1, 'run 1'), (axes[1], p2, 'run 2')):
    plot_data(ax, X_train, t_train)
    display_separator(ax, p.w, p.b)
    ax.set_title(f'{nom} — marge $\\approx$ {marge_separateur(p.w, p.b, X_train, t_train):.3f}')
for ax in axes:
    ax.set_xlim(-2.6, 2.6)
    ax.set_ylim(-2.6, 2.6)
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.set_aspect('equal')
    ax.grid(alpha=0.3)
plt.show()

m1 = marge_separateur(p1.w, p1.b, X_train, t_train)
m2 = marge_separateur(p2.w, p2.b, X_train, t_train)
err_train_1 = 1 - accuracy(t_train, np.array([p1.output(x) for x in X_train]))
err_train_2 = 1 - accuracy(t_train, np.array([p2.output(x) for x in X_train]))
err_test_1 = 1 - accuracy(t_test, np.array([p1.output(x) for x in X_test]))
err_test_2 = 1 - accuracy(t_test, np.array([p2.output(x) for x in X_test]))
print(f'run 1 : w = [{p1.w[0]:+.3f}, {p1.w[1]:+.3f}], b = {p1.b:+.3f}'
      f' | erreur train {err_train_1:.1%} | marge {m1:.3f} | erreurs de test {err_test_1:.1%}')
print(f'run 2 : w = [{p2.w[0]:+.3f}, {p2.w[1]:+.3f}], b = {p2.b:+.3f}'
      f' | erreur train {err_train_2:.1%} | marge {m2:.3f} | erreurs de test {err_test_2:.1%}')
# END

In [ ]:
assert err_train_1 == 0.0 and err_train_2 == 0.0, 'les deux runs doivent classer parfaitement le train'
assert m1 > m2, 'run 1 doit avoir la plus large marge'
assert abs(m1 - m2) > 0.03, 'les deux marges doivent être nettement différentes'
assert err_test_1 == 0.0, 'attendu : 0 erreur de test pour run 1'
assert np.isclose(err_test_2, 4 / 30), 'attendu : 4 erreurs de test sur 30 pour run 2'
print('comparaison check — OK')

> [!note] Constat
> Erreur d'entraînement : **0 % pour les deux runs** — elle ne distingue absolument pas les
> deux séparateurs. La marge, si : **≈ 0.276 contre ≈ 0.217** — et c'est le run à la plus
> large marge qui rate le moins le test (0 faute contre 4 sur 30).
>
> Reste à construire un algorithme qui **choisit** la grande marge au lieu de la subir —
> c'est le SVM, et on le code dès l'exercice suivant.